# Dataset EDA for Satellite Trail Segmentation

Exploratory analysis of the processed image subset and the supporting metadata catalogue. All reusable helpers live in `src/evaluation/eda.py` and `src/data/catalog.py`. Figures are written to `bg492/results/figures/` so they sit next to the rest of the run artefacts.

Sections:
1. Dataset overview tables
2. Full-frame image and mask inspection
3. Patch-level structure, sparsity, and trail examples
4. Mask completeness inspection (visual + quantitative undermasking check)
5. Metadata catalogue summary and visualisations
6. Observation date coverage (contextual; not a temporal trend)

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.data.catalog import SatelliteCatalog
from src.evaluation.eda import (
    compute_image_summary_dataframe,
    compute_mask_component_stats,
    compute_observation_date_dataframe,
    compute_patch_dataframe,
    plot_example_satellite_trail_patches,
    plot_image_level_summary,
    plot_mask_inspection_grid,
    plot_mask_thickness_distribution,
    plot_metadata_missing_values,
    plot_observation_date_distribution,
    plot_patch_density_distribution,
    plot_patch_density_heatmap,
    plot_ra_dec_distributions,
    plot_ra_dec_ranges,
    plot_random_image_mask_pairs,
    plot_random_mask_overlays,
    plot_satellite_name_frequency,
    plot_trail_length_distribution,
    summarise_patch_dataframe,
    summarise_patches_by_image,
)
from src.utils.logger import get_logger

In [ ]:
processed_dir = Path("../data/subset/processed")
catalog_path = Path("../data/subset/metadata/Satellites_Catalog_Application.csv")
figure_dir = Path("..") / "results" / "figures"
patch_size = 512
stride = 512
logger = get_logger("dataset_eda")

figure_dir.mkdir(parents=True, exist_ok=True)
processed_dir, catalog_path, figure_dir

## Dataset overview

Build the reusable summaries once, then reuse them throughout the notebook.

In [ ]:
catalog = SatelliteCatalog(catalog_path)
image_df = compute_image_summary_dataframe(processed_dir)
patch_df = compute_patch_dataframe(
    root_dir=processed_dir,
    patch_size=patch_size,
    stride=stride,
    logger=logger,
)
patch_summary_df = summarise_patches_by_image(patch_df)
dataset_stats = summarise_patch_dataframe(patch_df)

dataset_summary_df = pd.DataFrame([dataset_stats.to_dict()])
dataset_summary_df["empty_patch_fraction"] = (
    dataset_summary_df["empty_patches"] / dataset_summary_df["total_patches"]
)
dataset_summary_df["non_empty_patch_fraction"] = (
    dataset_summary_df["non_empty_patches"] / dataset_summary_df["total_patches"]
)

display(dataset_summary_df)
display(image_df)
display(patch_summary_df)
display(
    patch_df.loc[:, ["image_name", "y", "x", "positive_pixel_fraction", "is_empty"]]
    .sort_values("positive_pixel_fraction", ascending=False)
    .head(10)
)

## Full-frame image and mask inspection

In [ ]:
figure_paths = {}
figure_paths["random_pairs"] = plot_random_image_mask_pairs(
    root_dir=processed_dir,
    sample_count=2,
    seed=42,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["overlays"] = plot_random_mask_overlays(
    root_dir=processed_dir,
    sample_count=2,
    seed=42,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths

## Patch-level structure, sparsity, and trail examples

In [ ]:
figure_paths["image_level_summary"] = plot_image_level_summary(
    image_df=image_df,
    patch_df=patch_df,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["density_distribution"] = plot_patch_density_distribution(
    patch_df=patch_df,
    output_dir=figure_dir,
    logger=logger,
)

heatmap_paths = []
for image_name in patch_df["image_name"].unique():
    heatmap_paths.append(
        plot_patch_density_heatmap(
            patch_df=patch_df,
            image_name=image_name,
            output_dir=figure_dir,
            logger=logger,
        )
    )

figure_paths["heatmaps"] = heatmap_paths
figure_paths["trail_patches"] = plot_example_satellite_trail_patches(
    patch_df=patch_df,
    sample_count=6,
    sort_by_density=True,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths

## Mask completeness inspection

The supervisor flagged at the 2026-05-14 meeting that some overlays appear to under-cover the visible trail in the image. This section provides two diagnostics:

- A **shape distribution** over every connected component in the mask set — minor-axis (thickness) and aspect ratio. Long thin masks with very low minor-axis values are candidates for undermasking review.
- A **full-resolution inspection grid** of the densest non-empty patches, rendered at native 512×512 resolution so any trail pixel extending past the mask is visible.

Both figures are saved to `bg492/results/figures/`. Any quantitative undermasking response (dilation, re-annotation, accepted limitation) should follow visual confirmation here.

In [ ]:
component_df = compute_mask_component_stats(
    root_dir=processed_dir,
    logger=logger,
)
display(component_df.describe())
display(component_df.sort_values("aspect_ratio", ascending=False).head(10))

figure_paths["mask_thickness"] = plot_mask_thickness_distribution(
    component_df=component_df,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["mask_inspection_grid"] = plot_mask_inspection_grid(
    patch_df=patch_df,
    n_examples=6,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["mask_inspection_grid"]

## Metadata catalogue summary

In [ ]:
display(pd.DataFrame([catalog.get_metadata_summary()]))
display(catalog.get_missing_value_counts().to_frame(name="missing_values"))
display(catalog.get_numeric_summary())
display(catalog.get_satellite_frequency(top_n=15).to_frame(name="count"))
display(catalog.get_coordinate_dataframe().head(10))
catalog.get_ra_dec_ranges()

## Metadata catalogue visualisations

In [ ]:
figure_paths["satellite_frequency"] = plot_satellite_name_frequency(
    catalog=catalog,
    top_n=15,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["trail_length_distribution"] = plot_trail_length_distribution(
    catalog=catalog,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["metadata_missing_values"] = plot_metadata_missing_values(
    catalog=catalog,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["ra_dec_distributions"] = plot_ra_dec_distributions(
    catalog=catalog,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["ra_dec_ranges"] = plot_ra_dec_ranges(
    catalog=catalog,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths

## Observation date coverage

Dates are parsed from the `ML1_YYYYMMDD_HHMMSS` filename prefix. The labelled subset is not a uniform sample over time (it was selected for annotation, not as a survey snapshot) and we have no exposure-time metadata, so this figure is **strictly contextual** — it shows the temporal span of the training data and is not used to support any trend claim about trail incidence.

In [ ]:
date_df = compute_observation_date_dataframe(processed_dir)
display(date_df.head())
display(
    date_df["observation_year"]
    .value_counts()
    .sort_index()
    .to_frame(name="image_count")
)

figure_paths["observation_dates"] = plot_observation_date_distribution(
    date_df=date_df,
    output_dir=figure_dir,
    logger=logger,
)
figure_paths["observation_dates"]